# N14v5: Ceiling Breaker (Reverted — Double-Correction Bug Fixed)

**Executed N14v3 result (no manual prior-corr on TabPFN):**
- GBDT OOF 0.94997 | TabPFN OOF **0.94872** | Blend α=0.28 OOF **0.95031** | Public LB **0.95029**
- Beats N06 organic LB 0.95011. TabPFN is competitive once context ≥100K + compact features.

**N14v4 regressed this** by adding a manual `p /= class_priors` division on top of TabPFN's
already-set `balance_probabilities=True`. That is the exact "Double Correction" trap logged at
N07: when run in N15v4, this double-correction collapsed TabPFN OOF from 0.94872 to **0.93302**,
and the blender then dropped TabPFN entirely (weight 0.0). **This v5 reverts to N14v3's clean
recipe**: `balance_probabilities=True` only, no manual prior-division. See `notebooks/N17: Clean
Ceiling Blend/` for the actively-maintained successor with the same fix plus new levers.

Attach `prior-labsai/tabpfn-3`. GPU T4 x2.


In [ ]:
import os
os.environ["TABPFN_MODEL_CACHE_DIR"] = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
os.environ["TABPFN_NO_BROWSER"] = "1"
os.environ["TABPFN_MAX_BATCHED_TEST_ROWS"] = "4096"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install -q tabpfn 2>/dev/null || true


In [ ]:
import os, gc, time, warnings
import numpy as np, pandas as pd, torch, lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings("ignore")

ID_COL, TARGET_COL = "id", "health_condition"
N_FOLDS, SEED, N_CLASSES = 5, 42, 3
GPU_ENABLED = torch.cuda.is_available()
N_GPUS = torch.cuda.device_count() if GPU_ENABLED else 0
DEVICE = "cuda" if GPU_ENABLED else "cpu"
cb_task = "GPU" if GPU_ENABLED else "CPU"
print(f"GPU={GPU_ENABLED} | n_gpus={N_GPUS}")

_KAGGLE = "/kaggle/input/models/prior-labsai/tabpfn-3/pytorch/default/1"
TABPFN_CKPT = f"{_KAGGLE}/tabpfn-v3-classifier-v3_default.ckpt"
os.environ["TABPFN_MODEL_CACHE_DIR"] = _KAGGLE
os.environ["TABPFN_NO_BROWSER"] = "1"

TABPFN_OK = False
if os.path.isfile(TABPFN_CKPT):
    from tabpfn import TabPFNClassifier
    TABPFN_OK = True
    print(f"TabPFN checkpoint OK: {TABPFN_CKPT}")
else:
    print(f"MISSING: {TABPFN_CKPT}")

# NOTE: no prior_correct() here. TabPFNClassifier already gets balance_probabilities=True
# below; manually dividing by class priors on top of that is the N15v4 double-correction
# bug that collapsed TabPFN OOF from 0.94872 to 0.93302 (see N07 "double correction" trap).

def batch_predict(clf, X, bs=4096):
    X = np.asarray(X)
    return np.vstack([clf.predict_proba(X[i:i+bs]) for i in range(0, len(X), bs)])


In [ ]:
for p in ["/kaggle/input/competitions/playground-series-s6e7","/kaggle/input/playground-series-s6e7","../../data"]:
    if os.path.exists(os.path.join(p,"train.csv")):
        DATA_DIR=p; break
train_df=pd.read_csv(os.path.join(DATA_DIR,"train.csv"))
test_df=pd.read_csv(os.path.join(DATA_DIR,"test.csv"))
print(f"Train: {train_df.shape} | Test: {test_df.shape}")
le=LabelEncoder(); y=le.fit_transform(train_df[TARGET_COL])
class_priors = np.bincount(y, minlength=N_CLASSES).astype(np.float64) / len(y)
print(f"Class priors: {class_priors}")

cat_cols=["stress_level","physical_activity_level","diet_type","gender","smoking_alcohol","sleep_quality"]
num_cols=["sleep_duration","heart_rate","bmi","calorie_expenditure","step_count","exercise_duration","water_intake"]
all_cols=cat_cols+num_cols
X_str=train_df[all_cols].fillna("__NA__").astype(str)
X_str_te=test_df[all_cols].fillna("__NA__").astype(str)
imp=SimpleImputer(strategy="median")
X_num=imp.fit_transform(train_df[num_cols]); X_num_te=imp.transform(test_df[num_cols])
te=TargetEncoder(cv=5,target_type="multiclass",random_state=SEED)
X_te=te.fit_transform(X_str,y); X_te_test=te.transform(X_str_te)
X_full=np.hstack([X_num,X_te]); X_full_test=np.hstack([X_num_te,X_te_test])
print(f"TE matrix: {X_full.shape}")
folds=list(StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED).split(X_full,y))


In [ ]:
print("="*60+"\nGBDT ENSEMBLE + NUMERIC TE\n"+"="*60)
t0=time.time()
oof_g=np.zeros((len(train_df),N_CLASSES)); tst_g=np.zeros((len(test_df),N_CLASSES))
SEEDS=[42,2026,999]
for fold,(tr,va) in enumerate(folds):
    print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
    X_tr,X_va=X_full[tr],X_full[va]; y_tr=y[tr]
    sw=compute_sample_weight("balanced",y_tr)
    cc=np.bincount(y_tr,minlength=N_CLASSES); cb_w=[len(y_tr)/(N_CLASSES*c) for c in cc]
    fv=np.zeros((len(va),N_CLASSES)); ft=np.zeros((len(test_df),N_CLASSES)); nm=3*len(SEEDS)
    for s in SEEDS:
        m=CatBoostClassifier(iterations=1200,learning_rate=0.03,depth=6,class_weights=cb_w,random_seed=s,verbose=0,task_type=cb_task)
        m.fit(X_tr,y_tr,eval_set=(X_va,y[va]),early_stopping_rounds=100)
        fv+=m.predict_proba(X_va)/nm; ft+=m.predict_proba(X_full_test)/nm; del m
        lp=dict(n_estimators=1200,learning_rate=0.03,num_leaves=63,class_weight="balanced",random_state=s,n_jobs=-1,verbose=-1)
        if GPU_ENABLED: lp["device_type"]="gpu"
        m=lgb.LGBMClassifier(**lp)
        m.fit(X_tr,y_tr,eval_set=[(X_va,y[va])],callbacks=[lgb.early_stopping(100,verbose=False)])
        fv+=m.predict_proba(X_va)/nm; ft+=m.predict_proba(X_full_test)/nm; del m
        m=HistGradientBoostingClassifier(max_iter=1200,learning_rate=0.03,max_leaf_nodes=63,class_weight="balanced",random_state=s,early_stopping=True,validation_fraction=0.1,n_iter_no_change=100)
        m.fit(X_tr,y_tr,sample_weight=sw)
        fv+=m.predict_proba(X_va)/nm; ft+=m.predict_proba(X_full_test)/nm; del m
    oof_g[va]=fv; tst_g+=ft/N_FOLDS
    print(f"BAcc={balanced_accuracy_score(y[va],fv.argmax(1)):.5f}"); gc.collect()
score_g=balanced_accuracy_score(y,oof_g.argmax(1))
print(f">>> GBDT OOF: {score_g:.5f} ({time.time()-t0:.0f}s) <<<")


In [ ]:
print("="*60+"\nTabPFN-3 compact (balance_probabilities only, NO manual prior-division)\n"+"="*60)
oof_t=np.zeros((len(train_df),N_CLASSES)); tst_t=np.zeros((len(test_df),N_CLASSES)); score_t=0.0
if TABPFN_OK:
    X_tab=X_num.copy(); X_tab_te=X_num_te.copy()
    for c in cat_cols:
        cats=pd.Categorical(train_df[c].fillna("Missing").astype(str))
        X_tab=np.hstack([X_tab, cats.codes.astype(np.float64).reshape(-1,1)])
        te_codes=pd.Categorical(test_df[c].fillna("Missing").astype(str), categories=cats.categories).codes.astype(np.float64).reshape(-1,1)
        X_tab_te=np.hstack([X_tab_te, te_codes])
    devices=["cuda:0","cuda:1"] if N_GPUS>=2 else DEVICE
    try:
        t0=time.time()
        for fold,(tr,va) in enumerate(folds):
            print(f"  Fold {fold+1}/{N_FOLDS}...", end=" ", flush=True)
            sub,_=train_test_split(np.arange(len(tr)), train_size=min(100_000,len(tr)), stratify=y[tr], random_state=SEED+fold)
            clf=TabPFNClassifier(device=devices,n_estimators=2,balance_probabilities=True,fit_mode="fit_with_cache",model_path=TABPFN_CKPT,ignore_pretraining_limits=True)
            clf.fit(X_tab[tr[sub]], y[tr[sub]])
            va_p=batch_predict(clf,X_tab[va]); te_p=batch_predict(clf,X_tab_te)
            oof_t[va]=va_p; tst_t+=te_p/N_FOLDS
            print(f"BAcc={balanced_accuracy_score(y[va], va_p.argmax(1)):.5f}")
            del clf; gc.collect()
            if GPU_ENABLED: torch.cuda.empty_cache()
        score_t=balanced_accuracy_score(y, oof_t.argmax(1))
        print(f">>> TabPFN OOF: {score_t:.5f} ({time.time()-t0:.0f}s) <<<")
    except Exception as e:
        print(f"TabPFN FAILED: {e}"); score_t=0.0
else:
    print("TabPFN skipped")


In [ ]:
print("="*60+"\nBLEND\n"+"="*60)
best_score,best_name,best_probs=score_g,"GBDT Ensemble solo",tst_g
if score_t>0.90:
    for a in np.arange(0.0,1.01,0.02):
        sc=balanced_accuracy_score(y,(a*oof_t+(1-a)*oof_g).argmax(1))
        if sc>best_score:
            best_score=sc; best_name=f"Blend TabPFN={a:.2f}"; best_probs=a*tst_t+(1-a)*tst_g
    print(f"TabPFN solo: {score_t:.5f}")
print(f"GBDT solo:   {score_g:.5f}")
print(f">>> SELECTED: {best_name} | OOF {best_score:.5f} <<<")


In [ ]:
PSEUDO_THRESHOLD=0.99
mask=best_probs.max(1)>=PSEUDO_THRESHOLD
n_pseudo=int(mask.sum()); print(f"Pseudo-labels: {n_pseudo} ({100*n_pseudo/len(test_df):.1f}%)")
if n_pseudo>1000:
    py=best_probs[mask].argmax(1)
    aug_X=np.vstack([X_full,X_full_test[mask]]); aug_y=np.concatenate([y,py])
    aug_tst=np.zeros((len(test_df),N_CLASSES))
    for fold,(tr,va) in enumerate(StratifiedKFold(N_FOLDS,shuffle=True,random_state=SEED*2).split(aug_X,aug_y)):
        sw=compute_sample_weight("balanced",aug_y[tr])
        cc=np.bincount(aug_y[tr],minlength=N_CLASSES); cw=[len(aug_y[tr])/(N_CLASSES*c) for c in cc]
        fp=np.zeros((len(test_df),N_CLASSES))
        m=CatBoostClassifier(iterations=1200,learning_rate=0.03,depth=6,class_weights=cw,random_seed=SEED,verbose=0,task_type=cb_task)
        m.fit(aug_X[tr],aug_y[tr],eval_set=(aug_X[va],aug_y[va]),early_stopping_rounds=100)
        fp+=m.predict_proba(X_full_test)/3; del m
        lp=dict(n_estimators=1200,learning_rate=0.03,num_leaves=63,class_weight="balanced",random_state=SEED,n_jobs=-1,verbose=-1)
        if GPU_ENABLED: lp["device_type"]="gpu"
        m=lgb.LGBMClassifier(**lp)
        m.fit(aug_X[tr],aug_y[tr],eval_set=[(aug_X[va],aug_y[va])],callbacks=[lgb.early_stopping(100,verbose=False)])
        fp+=m.predict_proba(X_full_test)/3; del m
        m=HistGradientBoostingClassifier(max_iter=1200,learning_rate=0.03,max_leaf_nodes=63,class_weight="balanced",random_state=SEED,early_stopping=True,validation_fraction=0.1,n_iter_no_change=100)
        m.fit(aug_X[tr],aug_y[tr],sample_weight=sw)
        fp+=m.predict_proba(X_full_test)/3; del m
        aug_tst+=fp/N_FOLDS; print(f"  Aug fold {fold+1} done")
    final_probs=0.7*best_probs+0.3*aug_tst; best_name+=" + PseudoLabel"; print("Applied PL 70/30")
else:
    final_probs=best_probs


In [ ]:
preds=final_probs.argmax(1)
sub=pd.DataFrame({ID_COL:test_df[ID_COL].astype("int64"), TARGET_COL:le.inverse_transform(preds)})
sub.to_csv("submission.csv", index=False)
print("="*60+"\nN14v5 RESULTS (double-correction bug reverted)\n"+"="*60)
print(f"GBDT OOF:   {score_g:.5f}")
if score_t>0: print(f"TabPFN OOF: {score_t:.5f}")
print(f"Selected:   {best_name} | {best_score:.5f}")
print("Executed N14v3 LB=0.95029 | N06=0.95011 | v0.9=0.95065")
print(sub[TARGET_COL].value_counts())
